# EDA 10 (refined eda 8) — Case locator maps after EDA 7&9

**What changed vs the 20260704 version (and why):**

1. **Classification now comes from the EDA 9 time-symmetric mechanism tree.** 
   The old two-way split (genuine vs "exodus") mislabelled the 2011 outflow belt: 
   - **"exodus"** is an interpretation of the ***2021* external-majority** class, not a flow property that existed symmetrically in 2011. 
   - The tree replaces it with four measurable leaves applied identically to both years:

   | leaf | rule (per year *t*) |
   |---|---|
   | `inflow-driven`    | Frame-C Cascade-led ∧ inflow share ≥ 0.25 ∧ Frame-A Cascade-led |
   | `frame-sensitive`  | Frame-C Cascade-led ∧ inflow share ≥ 0.25 ∧ **not** frame-robust (residual) |
   | `outflow-external` | Frame-C Cascade-led ∧ inflow share < 0.25 ∧ ext arm share ≥ 0.5 |
   | `outflow-internal` | Frame-C Cascade-led ∧ inflow share < 0.25 ∧ ext arm share < 0.5 |

   Narrative labels live in captions only: 
   - *genuine cascade* = `inflow-driven` in both years (n = 8); 
   - *exodus* = the 2021 `outflow-external` class read against its 2011 baseline (19 → 78, ×4.1, 100 % outer London).

2. **Harrow 008/029 are reassigned.** 
   Their ext arm shares are 0.43/0.41 — majority of the downward outflow stays *within* London. They exemplify `outflow-internal` (the national-ladder artefact: from D8, 69 % of London counts as poorer), **not** the literal exodus. 
   
   **Kingston 616** (ext arm 0.69, national D9, inflow share 0.04) joins the case set as the `outflow-external` exemplar.

3. **Count tables by leaf × ring × year** 
   added for the Results chapter.

**Inputs:** `london_msoa_2011.geojson`, `eda9_mechanism_tree_20260705.csv`
<!-- (falls back to computing the tree inline from `eda4_results_for_phase3_20260626.csv` +
`msoa_cascade_national_frame_20260625.csv` if EDA 9 hasn't been run). -->

In [ ]:
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PolyCollection
from matplotlib.lines import Line2D
from shapely.geometry import shape
from shapely.ops import unary_union

plt.rcParams['font.family'] = 'DejaVu Sans'

In [ ]:
from pyprojroot import here

ROOT       = here()
sys.path.insert(0, str(ROOT))

DATA_DIR   = ROOT / 'data'
OUT_DIR    = ROOT / 'outputs' / 'case_study' / 'locator_map'
BOUNDARIES = DATA_DIR / 'london_msoa_2011.geojson'
EDA4       = ROOT / 'outputs' / 'eda4_results_for_phase3_20260626.csv'
NATF       = DATA_DIR / 'msoa_cascade_national_frame_20260625.csv'
EDA9       = ROOT / 'outputs' / 'eda9_mechanism_tree_20260705.csv'   # single interface from EDA 9
OUT_PNG    = OUT_DIR / 'fig01_case_locator_choropleth_v2.png'
OUT_PNG_2  = OUT_DIR / 'fig02_mechanism_geography_2011_2021_v2.png'

OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1. Palette and category styling

Mechanism names on the map; narrative labels in the caption. 

The two outflow leaves share the old exodus-pink family but 
- the **external-majority** shade is more saturated 
    - it carries the 2021 exodus reading); 
- **internal-majority** is paler (ladder artefact, not a departure story).

`frame-sensitive` gets amber so the residual is visible but clearly not part of either narrative.

In [ ]:
# C_IN   = '#c0392b'   # inflow-driven        — deep red   (gentrification signature)
# C_FS   = '#e67e22'   # frame-sensitive      — amber      (residual: reported, not narrated)
# # C_OE   = '#e8a7a0'   # outflow-external     — pale red   (exodus signature, 2021 reading)
# # C_OI   = '#f2cfc9'   # outflow-internal     — paler red  (Frame-B national-ladder artefact)
C_IN = '#99000d'   # inflow-driven      — deep red
C_OE = '#ef6548'   # outflow-external   — mid red-orange
C_OI = '#fcbba1'   # outflow-internal   — light pink
C_FS = '#e6ab02'   # frame-sensitive    — ochre/gold (clearly yellow vs the reds)
K_IN   = '#6a51a3'   # counter-led / flip   — purple
GREY   = '#e9e7e2'   # other                — grey fabric

CAT_STYLE = {
    'inflow-driven':    dict(fc=C_IN, alpha=0.95),
    'frame-sensitive':  dict(fc=C_FS, alpha=0.90),
    'outflow-external': dict(fc=C_OE, alpha=0.95),
    'outflow-internal': dict(fc=C_OI, alpha=0.95),
    'flip':             dict(fc=K_IN, alpha=0.80),   # fig 1 only (transition 2011→2021)
    'counter':          dict(fc=K_IN, alpha=0.60),   # fig 2 only (per-year state)
    'other':            dict(fc=GREY, alpha=1.00),
}

# bottom → top draw order (most salient last)
DRAW_FIG1 = ('other', 'flip', 'outflow-internal', 'outflow-external',
             'frame-sensitive', 'inflow-driven')
DRAW_FIG2 = ('other', 'counter', 'outflow-internal', 'outflow-external',
             'frame-sensitive', 'inflow-driven')

## 2. Case-study MSOAs

Harrow 008/029 stay in the set but as `outflow-internal` exemplars (see header note).

Kingston 616 is the new `outflow-external` exemplar. 

Label offsets for Kingston may need a nudge once plotted — same trial-and-error as the others.

In [ ]:
CASES = {
    'E02000191': 'Camden 026',        # inflow-driven, both years (genuine)
    'E02000873': 'Tower Hamlets 010', # inflow-driven, both years (genuine)
    'E02000809': 'Southwark 003',     # frame-sensitive 2011 → counter 2021
    'E02000561': 'Islington 008',     # inflow-driven 2011 → counter 2021 (flip)
    'E02000957': 'Wandsworth 035',    # inflow-driven 2011 → counter 2021 (flip)
    'E02000440': 'Harrow 008',        # outflow-internal (ladder artefact; inflow share 0.011)
    'E02000461': 'Harrow 029',        # outflow-internal (ladder artefact)
    'E02000616': 'Kingston 616',      # outflow-external (literal exodus; ext arm 0.69)
}

LABEL_OFF = {
    'Camden 026':        (-0.115,  0.016),
    'Tower Hamlets 010': ( 0.105,  0.026),
    'Southwark 003':     ( 0.095, -0.052),
    'Islington 008':     ( 0.090,  0.040),
    'Wandsworth 035':    (-0.035, -0.062),
    'Harrow 008':        ( 0.088,  0.032),
    'Harrow 029':        (-0.095, -0.034),
    'Kingston 616':      (-0.095, -0.045),
}

In [ ]:
INNER_LONDON_LADS = {
    'E09000007','E09000001','E09000011','E09000012','E09000013',
    'E09000014','E09000019','E09000020','E09000022','E09000023',
    'E09000025','E09000028','E09000030','E09000032','E09000033',
}

---
## 3. Mechanism classes — read from EDA 9 (or compute inline)

Primary path: the EDA 9 export (single-interface pattern). 

Fallback: identical tree computed inline, so this notebook also runs standalone. 

Thresholds are the same constants either way:
- `INFLOW_MIN = 0.25` (empirically gapped: 0.226→0.300 in 2011, 0.159→0.312 in 2021) and
- `EXT_MAJ = 0.5` (majority convention; sensitivity band reported in EDA 9 §4d).

In [ ]:
INFLOW_MIN = 0.25
EXT_MAJ    = 0.50
LEAVES     = ['inflow-driven', 'frame-sensitive', 'outflow-external', 'outflow-internal']

def _leaf(r, yr):
    if r[f'Typ_C_{yr}'] != 'Cascade-led':
        return None
    if r[f'Casc_Inflow_Share_{yr}'] >= INFLOW_MIN:
        return 'inflow-driven' if r[f'Typ_A_{yr}'] == 'Cascade-led' else 'frame-sensitive'
    return 'outflow-external' if r[f'Ext_Arm_Share_{yr}'] >= EXT_MAJ else 'outflow-internal'

if EDA9.exists():
    e9 = pd.read_csv(EDA9)
    # EDA 9 leaf names → short names used here
    rename = {'frame-sensitive inflow': 'frame-sensitive',
              'outflow: external-majority': 'outflow-external',
              'outflow: internal-majority': 'outflow-internal'}
    for yr in ('11', '21'):
        e9[f'leaf_{yr}'] = e9[f'leaf_{yr}'].replace(rename)
    print(f'loaded mechanism classes from {EDA9.name}')
else:
    e4 = pd.read_csv(EDA4)
    nf = pd.read_csv(NATF)
    e9 = e4.merge(nf[['msoa11cd',
                      'Outflow_Poorer_nat_11', 'Ext_Outflow_nat_11',
                      'Outflow_Poorer_nat_21', 'Ext_Outflow_nat_21']],
                  on='msoa11cd', validate='1:1')
    for yr in ('11', '21'):
        arm = e9[f'Outflow_Poorer_nat_{yr}'].replace(0, np.nan)
        e9[f'Ext_Arm_Share_{yr}'] = np.where(e9['Wealth_Decile_National'] > 6,
                                             e9[f'Ext_Outflow_nat_{yr}'] / arm, 0.0)
        e9[f'Ext_Arm_Share_{yr}'] = e9[f'Ext_Arm_Share_{yr}'].fillna(0.0)
        e9[f'leaf_{yr}'] = e9.apply(_leaf, axis=1, yr=yr)
    e9['Genuine_Persistent'] = ((e9['leaf_11'] == 'inflow-driven') &
                                (e9['leaf_21'] == 'inflow-driven'))
    print('EDA 9 export not found — computed mechanism tree inline (identical rules)')

In [ ]:
# ── figure-level categories ──────────────────────────────────────────────
# Fig 1 (2021-anchored single map): four 2021 leaves + flip (2011 cascade → 2021 counter) + other
def fig1_class(r):
    if isinstance(r['leaf_21'], str):
        return r['leaf_21']
    if r['Typ_C_11'] == 'Cascade-led' and r['Typ_C_21'] == 'Counter-led':
        return 'flip'
    return 'other'

# Fig 2 (per-year side-by-side): four leaves + counter + other, per year
def fig2_class(r, yr):
    if isinstance(r[f'leaf_{yr}'], str):
        return r[f'leaf_{yr}']
    return 'counter' if r[f'Typ_C_{yr}'] == 'Counter-led' else 'other'

e9['fig1'] = e9.apply(fig1_class, axis=1)
cat1 = dict(zip(e9['msoa11cd'], e9['fig1']))
cat1['E02000190'] = cat1.get('E02000189', 'other')          # Camden 024/025 merge
counts1 = e9['fig1'].value_counts()

cat2, counts2 = {}, {}
for yr in ('11', '21'):
    e9[f'fig2_{yr}'] = e9.apply(fig2_class, axis=1, yr=yr)
    d = dict(zip(e9['msoa11cd'], e9[f'fig2_{yr}']))
    d['E02000190'] = d.get('E02000189', 'other')
    cat2[yr] = d
    counts2[yr] = e9[f'fig2_{yr}'].value_counts()

print('fig1:', counts1.to_dict())
for yr in ('11', '21'):
    print(f'fig2 20{yr}:', counts2[yr].to_dict())

---
## 4. Count tables for the Results chapter

Leaf × year, and leaf × ring per year. External-majority is 100 % outer London in **both**
years; the inflow leaves are inner-weighted in 2011 (44 + 43 of 113 inner) and collapse by 2021.

In [ ]:
tab = pd.DataFrame({'2011': e9['leaf_11'].value_counts().reindex(LEAVES).fillna(0).astype(int),
                    '2021': e9['leaf_21'].value_counts().reindex(LEAVES).fillna(0).astype(int)})
tab.loc['cascade-led total'] = tab.sum()
tab['Δ'] = tab['2021'] - tab['2011']
print(tab.to_string(), end='\n\n')

for yr in ('11', '21'):
    print(f'20{yr} — leaf × ring')
    print(pd.crosstab(e9[f'leaf_{yr}'], e9['Ring'], margins=True).to_string(), end='\n\n')

if 'Genuine_Persistent' in e9.columns:
    print(f"Genuine cascade (inflow-driven in both years): {int(e9['Genuine_Persistent'].sum())} MSOAs")

---
## 5. Geometry

In [ ]:
gj = json.load(open(BOUNDARIES))
geoms, codes, lads = [], [], []
for ft in gj['features']:
    geoms.append(shape(ft['geometry']))
    codes.append(ft['properties']['MSOA11CD'])
    lads.append(ft['properties'].get('LAD11CD', ''))

def rings(geom):
    if geom.geom_type == 'Polygon':
        polys = [geom]
    elif geom.geom_type == 'MultiPolygon':
        polys = geom.geoms
    elif geom.geom_type == 'GeometryCollection':
        polys = [g for g in geom.geoms if g.geom_type == 'Polygon']
    else:
        polys = []
    return [np.asarray(p.exterior.coords) for p in polys]

london = unary_union(geoms)
inner  = unary_union([g for g, l in zip(geoms, lads) if l in INNER_LONDON_LADS])

In [ ]:
print(set(lads))          # if this prints {''} — that's the bug
print(inner.is_empty)     # will be True

In [ ]:
# after building geoms/codes — replace the lads list
lookup = pd.read_csv(DATA_DIR / 'MSOA_2011_to_2021_lookup_for_identification.csv',
                     encoding='utf-8-sig')          # first column has a BOM
msoa2lad = dict(zip(lookup['MSOA11CD'], lookup['LAD22CD']))
lads = [msoa2lad.get(c, '') for c in codes]

inner = unary_union([g for g, l in zip(geoms, lads) if l in INNER_LONDON_LADS])
assert not inner.is_empty, 'Inner London union came back empty — check LAD mapping'

---
## 6. Figure 1 — mechanism choropleth (2021-anchored) with case-study exemplars

Five mechanism classes + the cascade→counter flip band. Suggested caption:

> *Frame-C mechanism classification, 2021, with the 2011→2021 cascade→counter flip shown in
> purple. "Genuine cascade" (caption term) = inflow-driven; "exodus" = the external-majority
> outflow class, licensed by the −17.8 %/+51.3 % external-flow shift. Internal-majority
> outflow reflects the national-ladder artefact (from D8, most of London counts as poorer),
> not departure from London.*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# 1. choropleth, drawn per category so legend colours match exactly
for c in DRAW_FIG1:
    st = CAT_STYLE[c]
    polys = [r for g, code in zip(geoms, codes)
             if cat1.get(code, 'other') == c for r in rings(g)]
    if polys:
        ax.add_collection(PolyCollection(polys, facecolor=st['fc'],
                                         edgecolor='white', linewidth=0.3,
                                         alpha=st['alpha'], zorder=2))

# 2. Greater London + Inner London outlines
for r in rings(london):
    ax.plot(r[:, 0], r[:, 1], color='#6f6f6f', lw=1.1, zorder=5)
for r in rings(inner):
    ax.plot(r[:, 0], r[:, 1], color='#3d3d3d', lw=1.1, ls=(0, (5, 3)), zorder=5)

# 3. case MSOAs: black outline + leader label
for code, name in CASES.items():
    g = geoms[codes.index(code)]
    ax.add_collection(PolyCollection(rings(g), facecolor='none',
                                     edgecolor='black', linewidth=1.5, zorder=6))
    cx, cy = g.centroid.x, g.centroid.y
    dx, dy = LABEL_OFF[name]
    ax.annotate(name, xy=(cx, cy), xytext=(cx + dx, cy + dy),
                fontsize=8, fontweight='bold', color='#111',
                ha='center', va='center', zorder=8,
                arrowprops=dict(arrowstyle='-', color='#111', lw=0.8,
                                shrinkA=0, shrinkB=1),
                bbox=dict(boxstyle='round,pad=0.2', fc='white',
                          ec='#999', lw=0.5, alpha=0.9))

# 4. legend with live counts — mechanism names only
n = counts1
handles = [
    mpatches.Patch(fc=C_IN, ec='#666', lw=0.4, label=f"Inflow-driven cascade — frame-robust, inflow-led (n={n.get('inflow-driven',0)})"),
    mpatches.Patch(fc=C_FS, ec='#666', lw=0.4, label=f"Frame-sensitive inflow — Frame C only, residual (n={n.get('frame-sensitive',0)})"),
    mpatches.Patch(fc=C_OE, ec='#666', lw=0.4, label=f"Outflow cascade, external-majority (n={n.get('outflow-external',0)})"),
    mpatches.Patch(fc=C_OI, ec='#666', lw=0.4, label=f"Outflow cascade, internal-majority (n={n.get('outflow-internal',0)})"),
    mpatches.Patch(fc=K_IN, ec='#666', lw=0.4, alpha=0.8, label=f"Cascade \u2192 counter flip, 2011\u21922021 (n={n.get('flip',0)})"),
    mpatches.Patch(fc=GREY, ec='#666', lw=0.4, label=f"Other flow regimes (n={n.get('other',0)})"),
    Line2D([0], [0], color='#3d3d3d', lw=1.1, ls=(0, (5, 3)), label='Inner London (statutory)'),
    Line2D([0], [0], color='black', lw=1.5, label='Case-study MSOA'),
]
ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.01),
          ncol=2, fontsize=7.2, frameon=False, borderaxespad=0.0,
          handlelength=1.4, columnspacing=1.2)

ax.set_title('Flow mechanisms across London \u2014 EDA 9 tree, 2021 classes, with case-study exemplars',
             fontsize=11.5, fontweight='bold', pad=10)
ax.set_aspect(1.0 / np.cos(np.radians(51.5)))
ax.set_xlim(-0.60, 0.36); ax.set_ylim(51.26, 51.72)
ax.set_axis_off()

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=150, bbox_inches='tight')
print(f'saved {OUT_PNG}')
print(counts1.to_string())

---
## 7. Figure 2 — temporal side-by-side, same tree both years

Both panels now use the *identical* classification. 

The asymmetry the reader sees 
- deep red thinning out, 
- saturated pink flooding the outer belt — is an empirical finding, not a definitional artefact. 

Suggested caption:
> *Per-year mechanism classes, 2011 vs 2021. Inflow-driven cascades collapse 56→13;
> external-majority outflow cascades grow 19→78 (×4.1), all outer London — the pandemic-era
> exodus emerging against its 2011 baseline. Internal-majority outflow (paler) is the
> national-ladder artefact. Counter-led purple floods the inner core between panels.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 7.2))

for ax, yr, label_cases in zip(axes, ('11', '21'), (False, True)):
    for c in DRAW_FIG2:
        st = CAT_STYLE[c]
        polys = [r for g, code in zip(geoms, codes)
                 if cat2[yr].get(code, 'other') == c for r in rings(g)]
        if polys:
            ax.add_collection(PolyCollection(polys, facecolor=st['fc'],
                                             edgecolor='white', linewidth=0.25,
                                             alpha=st['alpha'], zorder=2))
    for r in rings(london):
        ax.plot(r[:, 0], r[:, 1], color='#6f6f6f', lw=1.0, zorder=5)
    for r in rings(inner):
        ax.plot(r[:, 0], r[:, 1], color='#3d3d3d', lw=1.0, ls=(0, (5, 3)), zorder=5)

    for code, name in CASES.items():
        g = geoms[codes.index(code)]
        ax.add_collection(PolyCollection(rings(g), facecolor='none',
                                         edgecolor='black', linewidth=1.4, zorder=6))
        if label_cases:
            cx, cy = g.centroid.x, g.centroid.y
            dx, dy = LABEL_OFF[name]
            ax.annotate(name, xy=(cx, cy), xytext=(cx + dx, cy + dy),
                        fontsize=7.4, fontweight='bold', color='#111',
                        ha='center', va='center', zorder=8,
                        arrowprops=dict(arrowstyle='-', color='#111', lw=0.7,
                                        shrinkA=0, shrinkB=1),
                        bbox=dict(boxstyle='round,pad=0.18', fc='white',
                                  ec='#999', lw=0.5, alpha=0.9))

    n = counts2[yr]
    ax.set_title(f'20{yr}', fontsize=13, fontweight='bold', pad=6)
    ax.text(0.5, -0.015,
            f"inflow-driven {n.get('inflow-driven',0)}   \u00b7   frame-sensitive {n.get('frame-sensitive',0)}   \u00b7   "
            f"outflow ext {n.get('outflow-external',0)}   \u00b7   outflow int {n.get('outflow-internal',0)}   \u00b7   "
            f"counter {n.get('counter',0)}   \u00b7   other {n.get('other',0)}",
            transform=ax.transAxes, ha='center', va='top', fontsize=8.6, color='#333')
    ax.set_aspect(1.0 / np.cos(np.radians(51.5)))
    ax.set_xlim(-0.60, 0.36); ax.set_ylim(51.26, 51.72)
    ax.set_axis_off()

handles = [
    mpatches.Patch(fc=C_IN, label='Inflow-driven cascade \u2014 frame-robust, inflow-led'),
    mpatches.Patch(fc=C_FS, label='Frame-sensitive inflow \u2014 residual'),
    mpatches.Patch(fc=C_OE, label='Outflow cascade \u2014 external-majority'),
    mpatches.Patch(fc=C_OI, label='Outflow cascade \u2014 internal-majority'),
    mpatches.Patch(fc=K_IN, alpha=0.6, label='Counter-led'),
    mpatches.Patch(fc=GREY, label='Symmetric / Lateral'),
    Line2D([0], [0], color='#3d3d3d', lw=1.0, ls=(0, (5, 3)), label='Inner London (statutory)'),
    Line2D([0], [0], color='black', lw=1.4, label='Case-study MSOA'),
]
fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False,
           fontsize=8.2, bbox_to_anchor=(0.5, 0.005))
fig.suptitle('Flow-mechanism geography of London, 2011 vs 2021 \u2014 time-symmetric tree (EDA 9)',
             fontsize=13.5, fontweight='bold', y=0.99)

plt.tight_layout(rect=[0, 0.085, 1, 0.965])
plt.savefig(OUT_PNG_2, dpi=150, bbox_inches='tight')
print(f'saved {OUT_PNG_2}')
for yr in ('11', '21'):
    print(f'20{yr}:', counts2[yr].to_dict())

---
### Quick interpretation (updated with verified EDA 9 numbers)

Both figures now read off the same time-symmetric tree, so every visual contrast between
panels is an empirical change, not a definitional one.

- **Genuine cascades** (inflow-driven, frame-robust) collapse **56 → 13**; only **8** MSOAs
  hold the class in both years — Camden ×2, Islington, Southwark ×2, Tower Hamlets, and
  (unexpectedly) Hillingdon ×2, worth a sentence in the case-study discussion.
- **The residual is symmetric and honest**: frame-sensitive inflow is **57 → 14**. In 2011
  over a quarter of the old "outflow-driven" belt was actually inflow-led but frame-fragile —
  the old two-way map overstated the 2011 outflow story.
- **The exodus emerges, it doesn't relocate**: external-majority outflow cascades grow
  **19 → 78 (×4.1)**, 100 % outer London in both years, spreading from 8 to 12 boroughs.
  Licensed as "exodus" by the −17.8 % / +51.3 % external-flow shift (EDA 9 §4a).
- **Internal-majority outflow** (81 → 90) is the stable backdrop: the national-ladder
  artefact, exemplified by Harrow 008/029 (ext arm 0.43/0.41, inflow share 0.011/0.038).
  Kingston 616 (ext arm 0.69) carries the literal-exodus exemplar role instead.
- **Counter-led purple floods the inner core between panels** (280 → 452) — unchanged
  from the previous version and still the third mechanism narrative.

Thresholds: 0.25 sits in an empirical gap both years (0.226→0.300; 0.159→0.312);
0.5 is a stated majority convention with the ±0.05 sensitivity band in EDA 9 §4d
(counts 38/19/8 in 2011 vs 103/78/46 in 2021 — growth ratio holds at every cut).